# 📊 06 - Data Input and Output

Most of data science is about **loading data, exploring it, and saving results**. Notebook 05 did this by hand with `open()`, `csv`, and `json`. Here, **Pandas** does the heavy lifting: one line to read a file into a table (a **DataFrame**) and one line to write it back out, in almost any format.

## ✅ Learning Goals
- Read CSV files (local or from a URL) into a DataFrame, and handle common variations with `read_csv` options
- Quickly inspect a DataFrame's shape, columns, types, and statistics
- Write CSV files without the "extra index column" mistake
- Read and write JSON in the common layouts, including nested data from APIs
- Read and write Excel workbooks, including multiple sheets
- **Convert** between CSV, JSON, and Excel, and spot what can go wrong along the way

## 0. Setup
Every file this notebook creates goes into a **`06-OutputFiles`** folder, through the `OUT` variable (the same pattern as notebook 05). Upload that folder along with this notebook when you submit.

In [ ]:
import pandas as pd
from pathlib import Path

OUT = Path('06-OutputFiles')
OUT.mkdir(exist_ok=True)

## 1. Reading CSV Files

In [ ]:
# Create a small CSV for demo
data = """Name,Age,Score
Alice,23,90
Bob,25,85
Charlie,22,95
"""
(OUT / 'students.csv').write_text(data, encoding='utf-8')

# Read CSV into a DataFrame
df = pd.read_csv(OUT / 'students.csv')
df

Compare this to notebook 05, where `csv.reader` gave you **strings** that you had to convert by hand. Pandas looks at each column and picks a type for you:

In [ ]:
df.dtypes

### Reading from a URL
`read_csv` can read straight from the web, which is how many public datasets are shared. (This needs an internet connection.)

In [ ]:
url = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv'
penguins = pd.read_csv(url)
penguins.head()

📶 **Offline?** If that cell fails, run `penguins = df` instead so the cells below still work (they'll just have less interesting output).

### Not Every CSV Looks the Same
Real files come with different separators, codes for missing values, and more columns or rows than you need. `read_csv` has options for all of these:

| Option | Use it when... | Example |
|---|---|---|
| `sep` | Columns are separated by something other than a comma | `sep=';'`, `sep='\t'` (tab) |
| `na_values` | Missing values are written as a code | `na_values=['N/A', '-', '?']` |
| `usecols` | You only need some of the columns | `usecols=['Name', 'Score']` |
| `nrows` | The file is huge and you just want a peek | `nrows=100` |
| `encoding` | You get a `UnicodeDecodeError` (common with files exported from Excel on Windows) | `encoding='latin-1'` |
| `dtype` | You need to force a column's type | `dtype={'Zip': str}` |

In [ ]:
messy = """Name;Age;Score
Dana;24;88
Eli;N/A;92
Fay;21;-
"""
(OUT / 'messy.csv').write_text(messy, encoding='utf-8')

messy_df = pd.read_csv(OUT / 'messy.csv', sep=';', na_values=['N/A', '-'])
messy_df

The missing values show up as `NaN` ("not a number"), which is Pandas' marker for missing data. Notice that `Age` and `Score` also turned into decimals (`24.0`): a number column with a missing value is stored as a float. Module 07 covers cleaning these up.

✅ **Your Turn**: Create your own small CSV (3–5 rows, at least one number column) at `OUT / 'my_data.csv'` and read it into a DataFrame called `my_df`.

In [ ]:
# Your turn here


## 2. Exploring DataFrames

A few attributes answer the first questions you'll have about any new dataset:

In [ ]:
print('shape:  ', penguins.shape)          # (rows, columns)
print('columns:', list(penguins.columns))

In [ ]:
# Look at the first rows
penguins.head()

`.tail()` shows the **last** rows, and `.sample(5)` shows 5 **random** rows, which is often more representative than the first few.

In [ ]:
# Info about columns, data types, and missing values
penguins.info()

In [ ]:
# Summary statistics for the number columns
penguins.describe()

💡 Read `.info()`'s **Non-Null Count** column carefully. If it's lower than the number of rows, that column has missing values. In `penguins`, several columns have a few.

✅ **Your Turn**: Use `.shape`, `.head()`, `.info()`, and `.describe()` on `my_df`.

In [ ]:
# Your turn here


## 3. Writing CSV Files

In [ ]:
# Save the DataFrame to a new CSV
df.to_csv(OUT / 'students_copy.csv', index=False)

print((OUT / 'students_copy.csv').read_text(encoding='utf-8'))

### Why `index=False`?
Every DataFrame has an **index**, the 0, 1, 2... row labels on the left. If you leave out `index=False`, Pandas writes the index as an extra unnamed column. Read the file back and you get a junk `Unnamed: 0` column:

In [ ]:
df.to_csv(OUT / 'students_with_index.csv')        # forgot index=False
pd.read_csv(OUT / 'students_with_index.csv')

✅ **Your Turn**: Save `my_df` to `OUT / 'my_data_copy.csv'` (without the index), then print the file's raw text to check it.

In [ ]:
# Your turn here


## 4. JSON Files
JSON is the language of web APIs and config files. A DataFrame can be written as JSON in several layouts, which Pandas calls **orients**. The one you'll want most often is `orient='records'`: a list with one dictionary per row, which is how most APIs return data.

In [ ]:
df.to_json(OUT / 'students.json', orient='records', indent=2)

print((OUT / 'students.json').read_text(encoding='utf-8'))

In [ ]:
pd.read_json(OUT / 'students.json')

⚠️ **The default orient is not `'records'`.** Without it, a DataFrame is saved **column by column**, keyed by the index. It's valid JSON, but awkward for anyone else to use:

In [ ]:
print(df.to_json(indent=2))

### JSON Lines
Large datasets and log files often use **JSON Lines** (`.jsonl`): one complete JSON object per line, with no surrounding list. Use `lines=True` to read or write it:

In [ ]:
df.to_json(OUT / 'students.jsonl', orient='records', lines=True)
print((OUT / 'students.jsonl').read_text(encoding='utf-8'))

pd.read_json(OUT / 'students.jsonl', lines=True)

### Nested JSON from APIs
API data is usually **nested**: dictionaries inside dictionaries. `pd.read_json` would leave the inner dictionaries crammed into single cells. `pd.json_normalize` **flattens** them into regular columns, joining the names with a dot:

In [ ]:
import json

api_response = [
    {'id': 1, 'name': 'Alice', 'contact': {'email': 'alice@example.com', 'city': 'Dallas'}},
    {'id': 2, 'name': 'Bob',   'contact': {'email': 'bob@example.com',   'city': 'Wichita Falls'}},
]
(OUT / 'api_users.json').write_text(json.dumps(api_response, indent=2), encoding='utf-8')

with open(OUT / 'api_users.json', encoding='utf-8') as f:
    raw = json.load(f)

users = pd.json_normalize(raw)
users

✅ **Your Turn**: Save `my_df` to `OUT / 'my_data.json'` using `orient='records'` and `indent=2`. Print the file's text, then read it back with `pd.read_json`.

In [ ]:
# Your turn here


## 5. Excel Files
Pandas reads and writes Excel through a separate package, `openpyxl`. Install it **once** with the `%pip` magic from notebook 01, which installs into this notebook's environment. Restart the kernel afterwards if Pandas still says it's missing.

In [ ]:
%pip install openpyxl

In [ ]:
# Save to Excel
df.to_excel(OUT / 'students.xlsx', index=False)

# Read the Excel file back
pd.read_excel(OUT / 'students.xlsx')

### Workbooks with Several Sheets
Real workbooks often have several sheets. To **write** several, use `pd.ExcelWriter`. To **read** them, use `sheet_name`:
- `sheet_name='Grades'`: one sheet, by name (`sheet_name=0` picks the first)
- `sheet_name=None`: **every** sheet, returned as a dictionary of `{sheet name: DataFrame}`

In [ ]:
with pd.ExcelWriter(OUT / 'school.xlsx') as writer:
    df.to_excel(writer, sheet_name='Grades', index=False)
    users.to_excel(writer, sheet_name='Contacts', index=False)

pd.read_excel(OUT / 'school.xlsx', sheet_name='Contacts')

In [ ]:
all_sheets = pd.read_excel(OUT / 'school.xlsx', sheet_name=None)

for sheet_name, sheet_df in all_sheets.items():
    print(f'{sheet_name}: {sheet_df.shape[0]} rows, {sheet_df.shape[1]} columns')

✅ **Your Turn**: Save `my_df` to `OUT / 'my_data.xlsx'` and read it back.

In [ ]:
# Your turn here


## 6. Converting Between Formats
Once data is in a DataFrame, converting it is always the same two steps: **read** with `pd.read_<format>()`, then **write** with `df.to_<format>()`.

| From \ To | CSV | JSON | Excel |
|---|---|---|---|
| **CSV** | | `read_csv` → `to_json` | `read_csv` → `to_excel` |
| **JSON** | `read_json` → `to_csv` | | `read_json` → `to_excel` |
| **Excel** | `read_excel` → `to_csv` | `read_excel` → `to_json` | |

### CSV → JSON

In [ ]:
pd.read_csv(OUT / 'students.csv').to_json(OUT / 'converted_students.json', orient='records', indent=2)

print((OUT / 'converted_students.json').read_text(encoding='utf-8'))

### JSON → CSV
For flat JSON, this is just `read_json` followed by `to_csv`. For **nested** JSON, flatten it with `json_normalize` first, because CSV can't hold a dictionary inside a cell:

In [ ]:
users.to_csv(OUT / 'converted_users.csv', index=False)

print((OUT / 'converted_users.csv').read_text(encoding='utf-8'))

### Excel → CSV (One File per Sheet)
CSV has no concept of "sheets", so a workbook becomes **one CSV per sheet**. Loop over the dictionary from `sheet_name=None`:

In [ ]:
for sheet_name, sheet_df in pd.read_excel(OUT / 'school.xlsx', sheet_name=None).items():
    out_file = OUT / f'school_{sheet_name}.csv'
    sheet_df.to_csv(out_file, index=False)
    print('Wrote', out_file)

### ⚠️ What Can Go Wrong in a Conversion
Formats don't store the same information, so a conversion can **quietly change your data**:
- **Leading zeros vanish.** CSV has no types, so `read_csv` turns ZIP code `'01234'` into the number `1234`. Fix it with `dtype={'Zip': str}`.
- **Formatting is lost.** Converting Excel → CSV drops colors, formulas (you get the computed values), and every sheet but one per file.
- **Nesting doesn't fit.** JSON → CSV needs `json_normalize`, or you get dictionaries crammed into cells.
- **The index sneaks in** whenever you forget `index=False`.

In [ ]:
(OUT / 'zips.csv').write_text("City,Zip\nBoston,02108\nDallas,75201\n", encoding='utf-8')

print(pd.read_csv(OUT / 'zips.csv'))                        # 02108 became 2108!
print()
print(pd.read_csv(OUT / 'zips.csv', dtype={'Zip': str}))    # kept as text

✅ **Your Turn**: Round-trip your data. Read `OUT / 'my_data.json'` (from section 4), save it as `OUT / 'my_data_roundtrip.csv'`, read **that** back, and check whether it matches the original with `my_df.equals(...)`. If it doesn't match, compare the `.dtypes` of both to find out why.

In [ ]:
# Your turn here


---
### Summary
- `pd.read_csv()` loads data into a DataFrame and picks column types for you. Use `sep`, `na_values`, `usecols`, `nrows`, `encoding`, and `dtype` for messy files. It also reads straight from a URL.
- `.shape`, `.columns`, `.dtypes`, `.head()`, `.info()`, and `.describe()` give you a quick first look.
- `df.to_csv(..., index=False)` saves results. Forgetting `index=False` gives you an `Unnamed: 0` column.
- For JSON, use `orient='records'` for a list of rows, `lines=True` for JSON Lines, and `pd.json_normalize()` to flatten nested API data.
- `to_excel`/`read_excel` need `openpyxl`. Use `pd.ExcelWriter` to write several sheets and `sheet_name=None` to read them all.
- Converting formats = `read_<format>` + `to_<format>`. Watch for lost leading zeros, lost formatting, nesting, and stray index columns.

## 🏋️ Practice

### Practice 1 — Select a Column (easy)
From `df`, print just the `'Score'` column, then print its average with `.mean()`. Do the same for the `'body_mass_g'` column of `penguins`. Does `.mean()` mind the missing values?

In [ ]:
# Your code here


### Practice 2 — Filter and Save (medium)
Filter `df` down to only students with a `Score` of 90 or above. Save that filtered DataFrame to `OUT / 'top_students.csv'` **and** to `OUT / 'top_students.json'` (as records).

In [ ]:
# Your code here


### Practice 3 — Read It Back and Verify (harder)
Read `OUT / 'top_students.csv'` back into a new DataFrame and confirm (with an `if`/`print`, not just by eyeballing it) that every row in it has a `Score` of 90 or above. *Hint:* `(new_df['Score'] >= 90).all()` is `True` only if **every** value passes. Then do the same check on `top_students.json`.

In [ ]:
# Your code here


## 🔥 Challenge (Optional) — A Loader for Any Format
Combine what you learned in Module 04 (More Functions & Loops), especially `safe_read_grades` in the files notebook, with this notebook's readers. Write a function `load_data(filename)` that:
1. Checks `Path(filename).suffix` first, and returns `None` with a message like `"Sorry, .txt files aren't supported"` for anything other than `.csv`, `.json`, or `.xlsx`.
2. Otherwise uses the suffix to pick the right reader (`.csv` → `pd.read_csv`, `.json` → `pd.read_json`, `.xlsx` → `pd.read_excel`) and returns the DataFrame.
3. Catches `FileNotFoundError` and returns `None` with a friendly message if the file doesn't exist.

**Bonus:** write `convert(src, dst)` that uses `load_data` to read `src` and picks the right `to_<format>` method from `dst`'s extension.

In [ ]:
def load_data(filename):
    # Your code here
    pass

load_data(OUT / 'students.csv')
load_data(OUT / 'students.json')
load_data(OUT / 'students.xlsx')
load_data(OUT / 'does_not_exist.csv')
load_data('notes.txt')

## 📦 Check Your Output Files
Run this last to see everything you created. All of it should be inside `06-OutputFiles`. Upload that folder along with this notebook.

In [ ]:
for path in sorted(OUT.rglob('*')):
    print(path)